# ExploraçõesSiss_Silver

In [5]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import IntegerType, DoubleType


# PARAMETERS

# Default. Quando correr pelo pipeline, este valor é substituído.
run_id = "manual"

print("Parâmetros recebidos pelo notebook:")
print("run_id:", run_id)


# 1. FUNÇÃO AUXILIAR DE LIMPEZA (Atualizada)
def clean_nan(col_name):
    """
    Converte strings 'NAN', 'nan', 'null' ou vazias em NULL real da base de dados.
    Aplica TRIM para remover espaços, mas MANTÉM A FORMATAÇÃO DE MAIÚSCULAS/MINÚSCULAS ORIGINAL.
    """
    c_original = F.trim(F.col(col_name))
    # Usamos o upper apenas para a condição de verificação, para apanhar "nan", "NaN", "NAN", etc.
    c_check = F.upper(c_original)
    
    return F.when(c_check.isin("NAN", "NULL", ""), F.lit(None)).otherwise(c_original)

# 2. CARREGAR DADOS BRONZE
df_brz = spark.read.table("brz.exploracoes_siss")


# 3. DEDUPLICAÇÃO (Manter apenas o registo mais recente por Marca)
window_spec = Window.partitionBy("marca") \
    .orderBy(F.desc("meta_source_file_date"), F.desc("audit_brz_load_timestamp"))

df_latest = df_brz.withColumn("row_num", F.row_number().over(window_spec)) \
    .filter(F.col("row_num") == 1) \
    .drop("row_num")


# 4. LIMPEZA, CASTING E PADRONIZAÇÃO
df_silver = df_latest.select(
    # Identificadores (Limpando NANs e APLICANDO UPPER APENAS NA MARCA)
    F.upper(clean_nan("marca")).alias("marca"),
    clean_nan("nif").alias("nif"),
    
    # Geografia (Tratamento especial: primeiro limpa NAN, depois troca vírgula por ponto)
    F.regexp_replace(clean_nan("localizacaolatitude"), ",", ".").cast(DoubleType()).alias("latitude"),
    F.regexp_replace(clean_nan("localizacaolongitude"), ",", ".").cast(DoubleType()).alias("longitude"),
    
    # Administrativo (Limpando NANs, mantendo texto original)
    clean_nan("distrito").alias("distrito"),
    clean_nan("concelho").alias("concelho"),
    clean_nan("freguesia_nome").alias("freguesia"),
    clean_nan("svl").alias("svl"),
    clean_nan("dsavr").alias("dsavr"),   
    clean_nan("freguesia_dicofre").alias("dicofre"),
    
    # Classificações Técnicas (Limpando NANs, mantendo texto original)
    clean_nan("tipo_de_entidade").alias("tipo_entidade"),
    clean_nan("tipo_de_instalacao").alias("tipo_instalacao"),
    clean_nan("tipo_de_exploracao").alias("tipo_exploracao"),
    clean_nan("sistema_de_exploracao").alias("sistema_exploracao"),
    clean_nan("tipo_de_producao").alias("tipo_producao"),
    clean_nan("modo_de_criacao").alias("modo_criacao"),
    clean_nan("estrutura_de_producao").alias("estrutura_producao"),
    
    # Datas (to_date ignora automaticamente strings inválidas como 'NAN' e devolve null)
    F.to_date(clean_nan("data_inicio")).alias("data_inicio_atividade"),
    F.to_date(clean_nan("data_fim")).alias("data_fim_atividade"),
    
    # Metadados
    clean_nan("periodo_da_ultima_des").alias("periodo_da_ultima_des"),
    F.col("meta_source_file").alias("meta_source_file"),
    F.col("meta_source_file_date").alias("meta_source_file_date"),
    F.current_timestamp().alias("audit_silver_refresh_timestamp"),
    F.lit(run_id).alias("audit_run_id")
)

# Remover linhas onde a marca é nula após a limpeza (lixo do Excel)
df_silver = df_silver.filter(F.col("marca").isNotNull())

# 5. CRIAÇÃO DE CHAVES AUXILIARES
df_silver = df_silver.withColumn(
    "marca_reduzida", 
    F.when(F.col("marca").startswith("PT"), F.substring(F.col("marca"), 3, 100))
     .otherwise(F.col("marca"))
)


# 6. GUARDAR NA SILVER
spark.sql("CREATE SCHEMA IF NOT EXISTS slv")

df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("slv.exploracoes_siss")

print(f"Sucesso! Tabela slv.exploracoes_siss gerada. Apenas a coluna 'marca' está em CAPS LOCK.")

StatementMeta(, 79760b41-7df6-4a4a-ac57-74e1bbcab68c, 7, Finished, Available, Finished, False)

Sucesso! Tabela slv.exploracoes_siss gerada. Apenas a coluna 'marca' está em CAPS LOCK.


## Validação

In [2]:
# from pyspark.sql import functions as F

# df = spark.read.table("slv.exploracoes_siss")

# print("Contagem de nulos reais vs Texto 'NAN' (deve ser 0):")
# df.select(
#     F.count(F.when(F.col("distrito").isNull(), 1)).alias("Nulos_Reais_Distrito"),
#     F.count(F.when(F.col("distrito") == "NAN", 1)).alias("Texto_NAN_Distrito")
# ).show()

# # Ver uma amostra de marcas com marca_reduzida
# display(df.select("marca", "marca_reduzida", "latitude", "longitude").limit(10))


# from pyspark.sql import functions as F

# # 1. Carregar a tabela Silver
# df = spark.read.table("slv.exploracoes_siss")

# print(f"{'='*80}")
# print(f"RELATÓRIO DE AUDITORIA PROFISSIONAL: slv.exploracoes_siss")
# print(f"{'='*80}\n")

# # --- 1. TESTE DE UNICIDADE (Deduplicação) ---
# # Se este número for > 0, o Join em Gold vai duplicar linhas (explosão cartesiana)
# duplicados = df.groupBy("marca").count().filter("count > 1").count()
# print(f"1. TESTE DE UNICIDADE:")
# print(f"   - Marcas duplicadas encontradas: {duplicados}")
# if duplicados == 0:
#     print("   - [OK] Unicidade garantida por marca.\n")
# else:
#     print("   - [ERRO] Atenção: Existem marcas repetidas na Silver!\n")

# # --- 2. INTEGRIDADE GEOGRÁFICA (Para Mapas) ---
# # Portugal está sensivelmente entre Lat: 32 a 42 e Long: -31 a -6
# print("2. AUDITORIA GEOGRÁFICA (Limites de Portugal):")
# geo_audit = df.select(
#     F.min("latitude").alias("min_lat"),
#     F.max("latitude").alias("max_lat"),
#     F.min("longitude").alias("min_long"),
#     F.max("longitude").alias("max_long")
# ).collect()[0]

# print(f"   - Latitude:  {geo_audit['min_lat']} até {geo_audit['max_lat']}")
# print(f"   - Longitude: {geo_audit['min_long']} até {geo_audit['max_long']}")

# # Verificar coordenadas impossíveis (ex: 0,0 ou fora do range)
# coordenadas_estranhas = df.filter(
#     (F.col("latitude") < 30) | (F.col("latitude") > 45) | 
#     (F.col("longitude") < -35) | (F.col("longitude") > -5)
# ).count()
# print(f"   - Registos com coordenadas fora do limite provável de Portugal: {coordenadas_estranhas}\n")

# # --- 3. AUDITORIA DE CAMPOS CRÍTICOS (Missing Data) ---
# print("3. QUALIDADE DE PREENCHIMENTO (Missing Values):")
# critical_cols = ["nif", "distrito", "concelho", "sistema_exploracao", "tipo_producao"]
# df.select([
#     F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in critical_cols
# ]).show()

# # --- 4. CONSISTÊNCIA TEMPORAL (Datas) ---
# print("4. VALIDAÇÃO DE DATAS:")
# # Atividade que termina antes de começar ou datas no futuro
# datas_invalidas = df.filter(
#     (F.col("data_fim_atividade") < F.col("data_inicio_atividade")) | 
#     (F.col("data_inicio_atividade") > F.current_date())
# ).count()
# print(f"   - Explorações com datas de atividade logicamente inconsistentes: {datas_invalidas}\n")

# # --- 5. ANÁLISE DE DISTRIBUIÇÃO (Top 5 por Distrito) ---
# print("5. DISTRIBUIÇÃO GEOGRÁFICA (Top 5 Distritos):")
# df.groupBy("distrito").count().orderBy(F.desc("count")).limit(5).show()

# # --- 6. TESTE DE COERÊNCIA NIF ---
# # Verifica se existem NIFs com menos de 9 dígitos (potencial erro de preenchimento)
# nif_curto = df.filter(F.length(F.col("nif")) < 9).count()
# print(f"6. QUALIDADE DE IDENTIFICAÇÃO:")
# print(f"   - NIFs com menos de 9 dígitos (possível erro): {nif_curto}")

# print(f"\n{'='*80}")
# print("AUDITORIA CONCLUÍDA")
# print(f"{'='*80}")

StatementMeta(, 79760b41-7df6-4a4a-ac57-74e1bbcab68c, 4, Finished, Available, Finished, False)

Contagem de nulos reais vs Texto 'NAN' (deve ser 0):
+--------------------+------------------+
|Nulos_Reais_Distrito|Texto_NAN_Distrito|
+--------------------+------------------+
|                   3|                 0|
+--------------------+------------------+



SynapseWidget(Synapse.DataFrame, 4d2db6e1-14b5-49a9-8127-946b492a6e5b)

RELATÓRIO DE AUDITORIA PROFISSIONAL: slv.exploracoes_siss

1. TESTE DE UNICIDADE:
   - Marcas duplicadas encontradas: 0
   - [OK] Unicidade garantida por marca.

2. AUDITORIA GEOGRÁFICA (Limites de Portugal):
   - Latitude:  -0.0 até 42.126365783333334
   - Longitude: -31.258099 até 0.0
   - Registos com coordenadas fora do limite provável de Portugal: 5

3. QUALIDADE DE PREENCHIMENTO (Missing Values):
+---+--------+--------+------------------+-------------+
|nif|distrito|concelho|sistema_exploracao|tipo_producao|
+---+--------+--------+------------------+-------------+
|  0|       3|       3|              8335|         5968|
+---+--------+--------+------------------+-------------+

4. VALIDAÇÃO DE DATAS:
   - Explorações com datas de atividade logicamente inconsistentes: 1

5. DISTRIBUIÇÃO GEOGRÁFICA (Top 5 Distritos):
+--------+-----+
|distrito|count|
+--------+-----+
|  Aveiro| 2408|
|   Braga| 2323|
| Coimbra| 2290|
|  Leiria| 2172|
|   Évora| 1470|
+--------+-----+

6. QUALIDADE D